# DataDose Initial Cleaning

Load raw DataDose dataset and perform initial standardization: drop unused columns, normalize text fields, and standardize dosage forms.

## 1. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Google Colab setup (uncomment if needed)
# from google.colab import drive
# drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DataDoseDepi'
INPUT_FILE = os.path.join(BASE_DIR, 'DataDoseDataset.csv')
OUTPUT_FILE = os.path.join(BASE_DIR, 'DataDoseDataset_Cleaned.csv')
LOG_FILE = os.path.join(BASE_DIR, 'cleaning_log.txt')

## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv(INPUT_FILE)

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nDuplicates: {df.duplicated().sum()}")

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

## 3. Initial Cleaning: Drop & Copy

In [ ]:
df_clean = df.copy()

cols_to_drop = ['updated', 'created', 'new_price', 'id', 'Therapeutic_Group']
cols_to_drop = [c for c in cols_to_drop if c in df_clean.columns]
df_clean = df_clean.drop(columns=cols_to_drop)

print(f"Dropped columns: {cols_to_drop}")
print(f"Remaining columns: {df_clean.columns.tolist()}")

## 4. Normalize Text Columns

In [ ]:
text_cols = ['activeingredient', 'company', 'form', 'group', 'route', 'tradename']

for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = (
            df_clean[col]
            .astype(str)
            .str.lower()
            .str.strip()
        )

print(f"Normalized text columns: {text_cols}")

## 5. Standardize Dosage Form

Collapse free-text dosage forms into consistent standard categories.

In [ ]:
if 'form' in df_clean.columns:
    df_clean['form'] = df_clean['form'].replace('nan', np.nan)
    df_clean = df_clean.dropna(subset=['form']).reset_index(drop=True)
    
    # Fix common typos
    form_typo_fix = {
        'caps': 'capsule',
        'cre': 'cream',
        'dro': 'drops',
        'power': 'powder',
        'tabs.': 'tablet',
        'tablets': 'tablet',
        'tab': 'tablet',
    }
    df_clean['form'] = df_clean['form'].replace(form_typo_fix)
    
    # Consolidation mappings
    form_consolidation = {
        'eye drops': 'drops',
        'ear drops': 'drops',
        'oral drops': 'drops',
        'nasal drops': 'drops',
        'mouth drops': 'drops',
        'ampoule': 'injection',
        'vial': 'injection',
        'syringe': 'injection',
        'pen': 'injection',
        'hair oil': 'topical',
        'conditioner': 'topical',
        'soap': 'topical',
        'shampoo': 'topical',
        'facial wash': 'topical',
        'paint': 'topical',
        'serum': 'topical',
        'oil': 'topical',
        'foam': 'topical',
        'gel': 'topical',
        'lotion': 'topical',
        'ointment': 'topical',
        'eye ointment': 'topical',
        'tablet': 'oral_solid',
        'lozenges': 'oral_solid',
        'film': 'oral_solid',
        'piece': 'oral_solid',
        'effervescent': 'oral_solid',
        'syrup': 'oral_liquid',
        'suspension': 'oral_liquid',
        'solution': 'oral_liquid',
        'mouth wash': 'oral_liquid',
        'vaginal douche': 'oral_liquid',
        'bottle': 'oral_liquid',
    }
    
    df_clean['form'] = df_clean['form'].replace(form_consolidation)
    df_clean['form_standardized'] = df_clean['form']
    
    print(f"Form value counts after standardization:")
    print(df_clean['form_standardized'].value_counts())
    print(f"\nNull values: {df_clean['form_standardized'].isnull().sum()}")

## 6. Summary & Export

In [ ]:
print(f"Final dataset shape: {df_clean.shape}")
print(f"Final columns: {df_clean.columns.tolist()}")
print(f"\nNull values:")
print(df_clean.isnull().sum())

df_clean.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved to: {OUTPUT_FILE}")